In [7]:
text = input("Nội dung: ")
with open("data.txt", "w", encoding="utf-8") as f:
    f.write(text)

!openssl enc -aes-256-cbc -pbkdf2 -in data.txt -out aes.bin -pass pass:123456
!openssl enc -aes-256-cbc -pbkdf2 -d -in aes.bin -out aes_dec.txt -pass pass:123456

!openssl genrsa -out rsa_priv.pem 2048 2>/dev/null
!openssl rsa -in rsa_priv.pem -pubout -out rsa_pub.pem 2>/dev/null
!openssl pkeyutl -encrypt -pubin -inkey rsa_pub.pem -in data.txt -out rsa.bin

!openssl ecparam -name prime256v1 -genkey -noout -out ec_priv.pem
!openssl ec -in ec_priv.pem -pubout -out ec_pub.pem 2>/dev/null
!openssl dgst -sha256 -sign ec_priv.pem -out sign.bin data.txt

print("\n[+] Hoàn thành")

Nội dung: sfsfsfdsfsdfdsf\

[+] Hoàn thành


In [3]:
!pip install cryptography

In [6]:
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding, ec

print("Part 2")


with open("rsa_priv.pem", "rb") as key_file:
    private_key = serialization.load_pem_private_key(key_file.read(), password=None)

with open("rsa.bin", "rb") as f:
    ciphertext = f.read()

plaintext = private_key.decrypt(
    ciphertext,
    padding.PKCS1v15()
)
print("[+] Kết quả Python giải mã RSA:")
print("=>", plaintext.decode('utf-8'), "\n")

with open("ec_pub.pem", "rb") as key_file:
    public_key = serialization.load_pem_public_key(key_file.read())

with open("sign.bin", "rb") as f:
    signature = f.read()
with open("data.txt", "rb") as f:
    original_data = f.read()

print("[+] Kết quả Python xác thực chữ ký ECDSA:")
try:

    public_key.verify(signature, original_data, ec.ECDSA(hashes.SHA256()))
    print("=> OK")
except Exception as e:
    print("=> NOT OK")

Part 2
[+] Kết quả Python giải mã RSA:
=> DNGUYENEENEE 

[+] Kết quả Python xác thực chữ ký ECDSA:
=> OK
